In [1]:
import numpy as np
import pandas as pd
import cmath
import math

def phasor(mag, ang_deg):
    return mag * cmath.exp(1j*math.radians(ang_deg))

def polar(z):
    return abs(z), math.degrees(cmath.phase(z))

def fmt(z, dec=3):
    m,a = polar(z)
    return f"{m:.{dec}f} ∠ {a:.{dec}f}°"

def seq_voltages(VAN, seq='pos'):
    
    if seq == 'pos':
        return VAN, VAN*cmath.exp(-1j*2*math.pi/3), VAN*cmath.exp(1j*2*math.pi/3)
    elif seq == 'neg':
        return VAN, VAN*cmath.exp(1j*2*math.pi/3), VAN*cmath.exp(-1j*2*math.pi/3)
    else:
        raise ValueError('seq deve ser pos ou neg')

def delta_from_y(Vy):
    # V_delta = sqrt(3)*Vy*exp(j*30deg)
    return Vy * math.sqrt(3) * cmath.exp(1j*math.radians(30))

def Z_parallel(*Zs):
    return 1/sum(1/z for z in Zs)


## Dados do problema

In [2]:
# Fontes
VAN_F1 = phasor(6350, 10)                 # F1: VAN (Y, seq +)
VAB_F2 = phasor(13800, 50)                # F2: VAB (Δ, seq -)
VAN_F2 = (VAB_F2 / math.sqrt(3)) * cmath.exp(-1j*math.radians(30))  # Δ→Y

# Linhas
Zp_L1_km = (0.15 + 0.20j)
l_L1 = 2.5
ZP1 = Zp_L1_km * l_L1                      # L1 total por fase

Zp_L2_km = (0.20 + 0.35j)
Zm_L2_km = 1j*0.09
l_L2 = 4.0
Zp_L2 = Zp_L2_km * l_L2
Zm_L2 = Zm_L2_km * l_L2
ZP2eq = Zp_L2 - Zm_L2                      # L2 equivalente por fase (equilibrado)

# Cargas
Zf_C1 = 18 + 6j
Zf_C2 = 12 - 4j
Zf_C3Y = (20/3) + 0j

print('VAN_F1 =', fmt(VAN_F1))
print('VAN_F2 (Δ→Y) =', fmt(VAN_F2))
print('ZP1 (L1) =', ZP1)
print('Zp_L2 =', Zp_L2, ' Zm_L2 =', Zm_L2, ' ZP2eq =', ZP2eq)

VAN_F1 = 6350.000 ∠ 10.000°
VAN_F2 (Δ→Y) = 7967.434 ∠ 20.000°
ZP1 (L1) = (0.375+0.5j)
Zp_L2 = (0.8+1.4j)  Zm_L2 = 0.36j  ZP2eq = (0.8+1.04j)


---

# Questão 1 — 12 combinações (F1/F2 × L1/L2 × C1/C2/C3)


In [3]:
def solve_q1(source='F1', line='L1', load='C1'):
    # Fonte
    if source == 'F1':
        VAN = VAN_F1; seq = 'pos'
    elif source == 'F2':
        VAN = VAN_F2; seq = 'neg'
    else:
        raise ValueError('Fonte inválida')

    V_AN, V_BN, V_CN = seq_voltages(VAN, seq)

    # Linha
    if line == 'L1':
        Zline = ZP1
        is_L2 = False
    elif line == 'L2':
        Zline = ZP2eq
        is_L2 = True
    else:
        raise ValueError('Linha inválida')

    # Carga
    if load == 'C1':
        Zf = Zf_C1; is_delta = False
    elif load == 'C2':
        Zf = Zf_C2; is_delta = False
    elif load == 'C3':
        Zf = Zf_C3Y; is_delta = True
    else:
        raise ValueError('Carga inválida')

    Zeq = Zline + Zf

    IA = V_AN/Zeq
    IB = V_BN/Zeq
    IC = V_CN/Zeq

    VAp = IA*Zf
    VBp = IB*Zf
    VCp = IC*Zf

    # Quedas
    if not is_L2:
        VAA = IA*ZP1; VBB = IB*ZP1; VCC = IC*ZP1
    else:
        # forma completa com mútuas
        VAA = IA*Zp_L2 + IB*Zm_L2 + IC*Zm_L2
        VBB = IA*Zm_L2 + IB*Zp_L2 + IC*Zm_L2
        VCC = IA*Zm_L2 + IB*Zm_L2 + IC*Zp_L2

    # Tensões Δ se C3
    if is_delta:
        VABp = delta_from_y(VAp)
        VBCp = delta_from_y(VBp)
        VCAp = delta_from_y(VCp)
    else:
        VABp = VBCp = VCAp = None

    return dict(source=source,line=line,load=load,
                IA=IA,IB=IB,IC=IC,
                VAp=VAp,VBp=VBp,VCp=VCp,
                VABp=VABp,VBCp=VBCp,VCAp=VCAp,
                VAA=VAA,VBB=VBB,VCC=VCC)

rows=[]
for F in ['F1','F2']:
    for L in ['L1','L2']:
        for C in ['C1','C2','C3']:
            r=solve_q1(F,L,C)
            rows.append({
                'Comb': f'{F}-{L}-{C}',
                'IA': fmt(r['IA']), 'IB': fmt(r['IB']), 'IC': fmt(r['IC']),
                "VA'N'": fmt(r['VAp']), "VB'N'": fmt(r['VBp']), "VC'N'": fmt(r['VCp']),
                "VAA'": fmt(r['VAA']), "VBB'": fmt(r['VBB']), "VCC'": fmt(r['VCC']),
                "VA'B'": '' if r['VABp'] is None else fmt(r['VABp']),
                "VB'C'": '' if r['VBCp'] is None else fmt(r['VBCp']),
                "VC'A'": '' if r['VCAp'] is None else fmt(r['VCAp']),
            })

df_q1 = pd.DataFrame(rows)
df_q1

,Comb,IA,IB,IC,VA'N',VB'N',VC'N',VAA',VBB',VCC',VA'B',VB'C',VC'A'
0,F1-L1-C1,325.795 ∠ -9.481°,325.795 ∠ -129.481°,325.795 ∠ 110.519°,6181.526 ∠ 8.954°,6181.526 ∠ -111.046°,6181.526 ∠ 128.954°,203.622 ∠ 43.649°,203.622 ∠ -76.351°,203.622 ∠ 163.649°,,,
1,F1-L1-C2,493.763 ∠ 25.792°,493.763 ∠ -94.208°,493.763 ∠ 145.792°,6245.659 ∠ 7.357°,6245.659 ∠ -112.643°,6245.659 ∠ 127.357°,308.602 ∠ 78.923°,308.602 ∠ -41.077°,308.602 ∠ -161.077°,,,
2,F1-L1-C3,899.510 ∠ 5.938°,899.510 ∠ -114.062°,899.510 ∠ 125.938°,5996.736 ∠ 5.938°,5996.736 ∠ -114.062°,5996.736 ∠ 125.938°,562.194 ∠ 59.069°,562.194 ∠ -60.931°,562.194 ∠ 179.069°,10386.651 ∠ 35.938°,10386.651 ∠ -84.062°,10386.651 ∠ 155.938°
3,F1-L2-C1,316.315 ∠ -10.529°,316.315 ∠ -130.529°,316.315 ∠ 109.471°,6001.663 ∠ 7.906°,6001.663 ∠ -112.094°,6001.663 ∠ 127.906°,415.037 ∠ 41.902°,415.037 ∠ -78.098°,415.037 ∠ 161.902°,,,
4,F1-L2-C2,483.338 ∠ 23.021°,483.338 ∠ -96.979°,483.338 ∠ 143.021°,6113.801 ∠ 4.586°,6113.801 ∠ -115.414°,6113.801 ∠ 124.586°,634.187 ∠ 75.452°,634.187 ∠ -44.548°,634.187 ∠ -164.548°,,,
5,F1-L2-C3,842.315 ∠ 2.071°,842.315 ∠ -117.929°,842.315 ∠ 122.071°,5615.433 ∠ 2.071°,5615.433 ∠ -117.929°,5615.433 ∠ 122.071°,1105.199 ∠ 54.502°,1105.199 ∠ -65.498°,1105.199 ∠ 174.502°,9726.216 ∠ 32.071°,9726.216 ∠ -87.929°,9726.216 ∠ 152.071°
6,F2-L1-C1,408.780 ∠ 0.519°,408.780 ∠ 120.519°,408.780 ∠ -119.481°,7756.047 ∠ 18.954°,7756.047 ∠ 138.954°,7756.047 ∠ -101.046°,255.487 ∠ 53.649°,255.487 ∠ 173.649°,255.487 ∠ -66.351°,,,
7,F2-L1-C2,619.531 ∠ 35.792°,619.531 ∠ 155.792°,619.531 ∠ -84.208°,7836.516 ∠ 17.357°,7836.516 ∠ 137.357°,7836.516 ∠ -102.643°,387.207 ∠ 88.923°,387.207 ∠ -151.077°,387.207 ∠ -31.077°,,,
8,F2-L1-C3,1128.628 ∠ 15.938°,1128.628 ∠ 135.938°,1128.628 ∠ -104.062°,7524.188 ∠ 15.938°,7524.188 ∠ 135.938°,7524.188 ∠ -104.062°,705.393 ∠ 69.069°,705.393 ∠ -170.931°,705.393 ∠ -50.931°,13032.277 ∠ 45.938°,13032.277 ∠ 165.938°,13032.277 ∠ -74.062°
9,F2-L2-C1,396.885 ∠ -0.529°,396.885 ∠ 119.471°,396.885 ∠ -120.529°,7530.370 ∠ 17.906°,7530.370 ∠ 137.906°,7530.370 ∠ -102.094°,520.752 ∠ 51.902°,520.752 ∠ 171.902°,520.752 ∠ -68.098°,,,


---

# Questão 2 — cargas em paralelo (3 combinações)

1. F1–L1–(C1//C2)
2. F2–L2–(C1//C3)
3. F1–L2–(C1//C2//C3)


In [4]:
def solve_q2(case_id):
    if case_id == 1:
        VAN = VAN_F1; seq='pos'; Zline=ZP1
        Zfeq = Z_parallel(Zf_C1, Zf_C2)
        is_L2 = False
    elif case_id == 2:
        VAN = VAN_F2; seq='neg'; Zline=ZP2eq
        Zfeq = Z_parallel(Zf_C1, Zf_C3Y)
        is_L2 = True
    elif case_id == 3:
        VAN = VAN_F1; seq='pos'; Zline=ZP2eq
        Zfeq = Z_parallel(Zf_C1, Zf_C2, Zf_C3Y)
        is_L2 = True
    else:
        raise ValueError('case_id inválido')

    V_AN, V_BN, V_CN = seq_voltages(VAN, seq)
    Zeq = Zline + Zfeq

    IA = V_AN/Zeq
    IB = V_BN/Zeq
    IC = V_CN/Zeq

    VAp = IA*Zfeq
    VBp = IB*Zfeq
    VCp = IC*Zfeq

    # Quedas
    if not is_L2:
        VAA = IA*ZP1; VBB = IB*ZP1; VCC = IC*ZP1
    else:
        VAA = IA*Zp_L2 + IB*Zm_L2 + IC*Zm_L2
        VBB = IA*Zm_L2 + IB*Zp_L2 + IC*Zm_L2
        VCC = IA*Zm_L2 + IB*Zm_L2 + IC*Zp_L2

    # Δ tensão (apenas para referência quando C3 estiver presente)
    VABp = delta_from_y(VAp)
    VBCp = delta_from_y(VBp)
    VCAp = delta_from_y(VCp)

    return dict(IA=IA,IB=IB,IC=IC,VAp=VAp,VBp=VBp,VCp=VCp,VABp=VABp,VBCp=VBCp,VCAp=VCAp,VAA=VAA,VBB=VBB,VCC=VCC)

labels={1:'F1-L1-C1//C2',2:'F2-L2-C1//C3',3:'F1-L2-C1//C2//C3'}
rows=[]
for k in [1,2,3]:
    r=solve_q2(k)
    rows.append({
        'Comb': labels[k],
        'IA': fmt(r['IA']), 'IB': fmt(r['IB']), 'IC': fmt(r['IC']),
        "VA'N'": fmt(r['VAp']), "VB'N'": fmt(r['VBp']), "VC'N'": fmt(r['VCp']),
        "VAA'": fmt(r['VAA']), "VBB'": fmt(r['VBB']), "VCC'": fmt(r['VCC']),
        "VA'B'": fmt(r['VABp']), "VB'C'": fmt(r['VBCp']), "VC'A'": fmt(r['VCAp']),
    })

df_q2 = pd.DataFrame(rows)
df_q2

,Comb,IA,IB,IC,VA'N',VB'N',VC'N',VAA',VBB',VCC',VA'B',VB'C',VC'A'
0,F1-L1-C1//C2,761.422 ∠ 10.213°,761.422 ∠ -109.787°,761.422 ∠ 130.213°,6077.885 ∠ 6.399°,6077.885 ∠ -113.601°,6077.885 ∠ 126.399°,475.889 ∠ 63.343°,475.889 ∠ -56.657°,475.889 ∠ -176.657°,10527.205 ∠ 36.399°,10527.205 ∠ -83.601°,10527.205 ∠ 156.399°
1,F2-L2-C1//C3,1339.969 ∠ 5.848°,1339.969 ∠ 125.848°,1339.969 ∠ -114.152°,6676.704 ∠ 10.611°,6676.704 ∠ 130.611°,6676.704 ∠ -109.389°,1758.170 ∠ 58.279°,1758.170 ∠ 178.279°,1758.170 ∠ -61.721°,11564.390 ∠ 40.611°,11564.390 ∠ 160.611°,11564.390 ∠ -79.389°
2,F1-L2-C1//C2//C3,1401.918 ∠ -1.847°,1401.918 ∠ -121.847°,1401.918 ∠ 118.153°,5095.543 ∠ -3.583°,5095.543 ∠ -123.583°,5095.543 ∠ 116.417°,1839.453 ∠ 50.584°,1839.453 ∠ -69.416°,1839.453 ∠ 170.584°,8825.740 ∠ 26.417°,8825.740 ∠ -93.583°,8825.740 ∠ 146.417°


---

# Questão 3 — Duas fontes


In [5]:

VAN  = phasor(6350, 10)     # Fonte F1
VA2N = phasor(6600, 15)     # Fonte F3


ZP1 = ZP1                   # Linha L1
ZP2 = ZP2eq                 # Linha L2 equivalente (zp2 = zp - zm)
Zf  = Zf_C1                 # Carga C1


Yeq = (1/Zf + 1/ZP1 + 1/ZP2)
VAp = (VAN/ZP1 + VA2N/ZP2) / Yeq


# Correntes fase A

IA  = (VAN  - VAp)/ZP1      # Fonte F1
IA2 = (VA2N - VAp)/ZP2      # Fonte F3
IAp = VAp/Zf                # Carga


# Sistema equilibrado (seq. positiva)

a = cmath.exp(1j*2*math.pi/3)

IB,  IC   = IA * a.conjugate(),  IA * a
IB2, IC2  = IA2 * a.conjugate(), IA2 * a
IBp, ICp  = IAp * a.conjugate(), IAp * a

VBp, VCp  = VAp * a.conjugate(), VAp * a


# Quedas de tensão

VAA_L1 = IA  * ZP1
VBB_L1 = IB  * ZP1
VCC_L1 = IC  * ZP1

VAA_L2 = IA2 * ZP2
VBB_L2 = IB2 * ZP2
VCC_L2 = IC2 * ZP2


rows = [
    {"Grandeza": "IA (F1) [A]",     "Valor": fmt(IA)},
    {"Grandeza": "IB (F1) [A]",     "Valor": fmt(IB)},
    {"Grandeza": "IC (F1) [A]",     "Valor": fmt(IC)},

    {"Grandeza": "IA'' (F3) [A]",   "Valor": fmt(IA2)},
    {"Grandeza": "IB'' (F3) [A]",   "Valor": fmt(IB2)},
    {"Grandeza": "IC'' (F3) [A]",   "Valor": fmt(IC2)},

    {"Grandeza": "IA' (carga) [A]", "Valor": fmt(IAp)},
    {"Grandeza": "IB' (carga) [A]", "Valor": fmt(IBp)},
    {"Grandeza": "IC' (carga) [A]", "Valor": fmt(ICp)},

    {"Grandeza": "VA'N' [V]",       "Valor": fmt(VAp)},
    {"Grandeza": "VB'N' [V]",       "Valor": fmt(VBp)},
    {"Grandeza": "VC'N' [V]",       "Valor": fmt(VCp)},

    {"Grandeza": "VAA' (L1) [V]",   "Valor": fmt(VAA_L1)},
    {"Grandeza": "VBB' (L1) [V]",   "Valor": fmt(VBB_L1)},
    {"Grandeza": "VCC' (L1) [V]",   "Valor": fmt(VCC_L1)},

    {"Grandeza": "VA''A' (L2) [V]", "Valor": fmt(VAA_L2)},
    {"Grandeza": "VB''B' (L2) [V]", "Valor": fmt(VBB_L2)},
    {"Grandeza": "VC''C' (L2) [V]", "Valor": fmt(VCC_L2)},
]

df_q3 = pd.DataFrame(rows)
df_q3

,Grandeza,Valor
0,IA (F1) [A],181.397 ∠ -110.480°
1,IB (F1) [A],181.397 ∠ 129.520°
2,IC (F1) [A],181.397 ∠ 9.520°
3,IA'' (F3) [A],412.949 ∠ 17.858°
4,IB'' (F3) [A],412.949 ∠ -102.142°
5,IC'' (F3) [A],412.949 ∠ 137.858°
6,IA' (carga) [A],332.419 ∠ -7.484°
7,IB' (carga) [A],332.419 ∠ -127.484°
8,IC' (carga) [A],332.419 ∠ 112.516°
9,VA'N' [V],6307.208 ∠ 10.951°
